# Source Conflict Resolution — Documentation

This notebook documents the decisions taken on all source conflicts with a percentage discrepancy above 100% in `NR_source_conflicts.csv`. Each conflict was investigated by cross-referencing the OWID readme documentation (which cites USGS MCS 2025 and BGS 2025) against the EI Statistical Review data.

**Pipeline reference:** `0_NR_extraction_FINAL.ipynb`  
**Conflict log:** `NR_source_conflicts.csv` (5,631 rows total; 18 rows above 100%)

---

## Summary of conflicts above 100%

| Resource | Countries affected | Metric | Discrepancy range | Root cause | Resolution |
|---|---|---|---|---|---|
| Natural Graphite | Turkey, India, Madagascar | Production | 132–950% | Commodity scope mismatch (OWID = natural graphite content per USGS; EI = broader definition, likely including ore or synthetic) | **Keep OWID.** Already the default via `_MINERAL_PROD` priority. |
| Natural Graphite | Turkey | Reserves | 900% | OWID = 6,900 kt, EI = 69,000 kt. Both claim USGS sourcing. Likely a processing error in one source or a reserves-vs-resources distinction. | **Keep OWID** (default). Flag for manual verification against USGS MCS 2025 Table. |
| Platinum Group | Russia | Reserves | 294% | OWID reports **palladium-only** reserves (readme title: "Platinum group metals (palladium) reserves"). EI reports **total PGM** reserves. USGS MCS note: Russia uses A+B+C1+C2 classification. | **Keep OWID** (default). Acceptable since pipeline tracks PGMs as a group and OWID/USGS is the more conservative, well-documented estimate. |
| Rare Earth | Madagascar | Production | 171% (x2) | OWID (USGS) reports **REO equivalent content**. EI likely reports ore tonnage. USGS readme explicitly states: "World production data were for REO equivalent content of ores produced." | **Keep OWID.** Already the default. The duplicate (vs EI_CSV and vs EI_Excel) confirms both EI sources agree on the higher, likely ore-based figure. |
| Coal | Mexico | Production | 103–167% | Historical (1981–82). Likely hard coal vs total coal (incl. lignite). | **No action.** Low priority, historical, does not affect classification. |

---

## 1. Natural Graphite — Production

**Affected rows:** Turkey (2020–2024), India (2020–2024), Madagascar (2020)  
**Discrepancy range:** 132% to 950%

### Evidence

The OWID graphite production readme states:
- Unit: **tonnes**
- Source: USGS MCS 2025 + USGS Historical Statistics 2024
- Scope: *"Production of graphite, in tonnes. Based on mined, rather than refined, production."*
- BGS note: *"This table includes all forms of amorphous and crystalline graphite but excludes synthetic material."*

After the pipeline applies `OWID_TO_CANONICAL` (dividing by 1,000 to convert tonnes to kt), OWID values for Turkey sit at 2.5–3.1 kt, consistent with USGS MCS reports of Turkish natural graphite output.

EI values for the same country-years are 15–28 kt, roughly an order of magnitude higher. This pattern repeats for India (OWID 6–28 vs EI 30–130 kt) and Madagascar (OWID 21 vs EI 48.5 kt).

### Diagnosis

The discrepancy is not a units error. Both sources are in kt after conversion. The difference is in **commodity scope**: OWID/USGS reports natural graphite mineral content, while EI appears to report a broader aggregate (possibly including beneficiated material, ore weight, or a different product-stage definition).

### Decision

**Keep OWID (USGS).** This is already the default behaviour under `_MINERAL_PROD = ['OWID', 'EI_Excel', 'EI_CSV', 'EI_Prices']`. No pipeline change required.

## 2. Natural Graphite — Reserves (Turkey 2024)

**Affected rows:** Turkey 2024  
**Discrepancy:** 900% (OWID = 6,900 kt; EI_Excel = 69,000 kt)

### Evidence

The OWID graphite reserves readme states:
- Unit: **tonnes**
- Source: USGS MCS 2025 only (no BGS for reserves)
- Scope: *"Reserves of graphite, in tonnes. Mineral reserves are resources that have been evaluated and can be mined economically with current technologies."*

After unit conversion (OWID tonnes / 1,000 = kt), OWID reports 6,900 kt = 6.9 Mt for Turkey. EI reports 69,000 kt = 69 Mt. The USGS MCS 2025 typically lists Turkey's graphite reserves around 90 Mt, suggesting neither source aligns perfectly. EI is closer to the raw USGS figure; OWID may have a processing error or may be reporting a sub-category.

### Diagnosis

The factor-of-10 difference (6.9 vs 69) suggests one source may have applied an extra /1,000 conversion, or OWID is reporting a narrower reserves category. This cannot be fully resolved without inspecting the raw USGS MCS data table.

### Decision

**Keep OWID** (default under `_MINERAL_RES`). The conflict is flagged for manual verification against the raw USGS MCS 2025 graphite table. If OWID is confirmed to have a processing error, consider overriding to EI_Excel for this specific mineral-metric pair.

## 3. Platinum Group Metals — Reserves (Russia 2024)

**Affected rows:** Russia 2024  
**Discrepancy:** 294% (OWID = 16 kt; EI_Excel = 63 kt)

### Evidence

The OWID PGM reserves readme states:
- Unit: **tonnes**
- Source: USGS MCS 2025
- Scope: explicitly titled *"Platinum group metals (palladium) reserves"*
- Note on Russia: *"Reserves for Russia are based on the Russian Classification system A+B+C1+C2, where C2 are deposits that are being developed or prepared for development."*

This means OWID is reporting **palladium-only** reserves, not total PGM. EI_Excel's "Platinum Group Metals P-R" sheet likely reports reserves across all six platinum group metals (platinum, palladium, rhodium, ruthenium, iridium, osmium).

### Diagnosis

**Commodity scope mismatch.** Palladium-only (OWID) vs total PGM (EI). The 4x ratio (16 vs 63) is consistent with palladium typically accounting for a fraction of total PGM reserves.

### Decision

**Keep OWID** (default under `_MINERAL_RES`). Note that the pipeline already handles PGM production separately with `('Platinum Group', 'Production'): ['EI_Excel', 'OWID', ...]`, preferring EI for production. For reserves, OWID (palladium-only) is kept as the default. If the capstone requires total PGM reserves rather than palladium-only, the priority for `('Platinum Group', 'Reserves')` should be changed to `['EI_Excel', 'OWID', ...]`.

## 4. Rare Earth — Production (Madagascar 2022)

**Affected rows:** Madagascar 2022 (appears twice: once vs EI_Excel, once vs EI_CSV)  
**Discrepancy:** 171% in both cases (OWID = 0.96 kt; EI = 2.6 kt)

### Evidence

The OWID rare earths production readme states:
- Unit: **tonnes**
- Source: USGS MCS 2025 + USGS Historical Statistics 2024
- USGS note: *"World production data were for REO equivalent content of ores produced."*

REO (rare earth oxide) equivalent content is the standard USGS reporting convention for rare earths. It represents the refined oxide weight extractable from ore, which is substantially less than the gross ore tonnage.

### Diagnosis

OWID reports 0.96 kt (960 tonnes) of REO content, consistent with USGS MCS 2022 data for Madagascar. Both EI sources report 2.6 kt, likely measuring ore rather than oxide content. The fact that EI_CSV and EI_Excel agree on 2.6 confirms this is a consistent alternative definition rather than a data error.

### Decision

**Keep OWID (USGS REO content).** Already the default under `_MINERAL_PROD`. No pipeline change required. The ore-vs-content distinction is documented but does not require intervention since OWID/USGS provides the more standardised and widely used measure.

## 5. Coal — Production (Mexico 1981–82)

**Affected rows:** Mexico 1981, Mexico 1982  
**Discrepancy:** 167% and 103%

### Evidence

No OWID readme was consulted for coal production. The discrepancy (EI_CSV ~3.0–3.75 Mt vs OWID ~7.6–8.1 Mt) is confined to two historical years and likely reflects a difference in coal type coverage (hard coal only vs hard coal + lignite, or different calorific-value adjustments).

### Decision

**No action.** Historical edge case with negligible impact on the resource dependency classification. The default priority (`_EI_HYDRO = ['EI_CSV', 'OWID', ...]`) keeps EI_CSV for coal production, so the lower value is retained.

---

## Action items

1. **Verify Turkey graphite reserves** against the raw USGS MCS 2025 graphite commodity table. If OWID's 6.9 Mt is confirmed as a processing error, add an explicit override in `RESOURCE_PRIORITY` for `('Natural Graphite', 'Reserves')`.
2. **Decide on PGM scope for the capstone.** If total PGM reserves (not palladium-only) are needed for the dependency classification, add `('Platinum Group', 'Reserves'): ['EI_Excel', 'OWID', 'EI_CSV', 'EI_Prices']` to `RESOURCE_PRIORITY`.
3. **No code changes needed** for graphite production, rare earth production, or coal production conflicts. The existing priority hierarchy and OWID/USGS sourcing are correct.
4. **All conflicts below 100%** (5,613 of 5,631 rows) are within acceptable ranges given known differences in rounding, vintage, and minor definitional variation across sources.